# Narrowing down variant sites causative to stem juiciness phenotype
Assumed to be found on SBC4, 10, and 23 exclusively, based on the association with the phenotype.

In [2]:
# open vcf file
import numpy as np
import cyvcf2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gzip, re
from pathlib import Path

sns.set_theme(style="whitegrid")

SAMPLE   = "SBC4, 10, and 23"
VCF_PATH = "../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz"
GFF_PATH = "../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz"

## Overall steps
1. Prepare dataframe
2. Scoring variant sites based on its impact & risk score
3. Convert variant sites into gene using 4 different approaches

These steps executed by the following script

In [3]:
!../docker/run.sh python3 ../workflow/scripts/genomics_scoring.py -i ../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz -o ../results/gene_score/ -g ../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz

[genomics_scoring] Parsing VCF: ../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz
[genomics_scoring]   205,996 (variant, gene) rows
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.variant.tsv  (205,996 variant annotations)
[genomics_scoring] Loading gene lengths: ../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz
[genomics_scoring]   32,458 genes with length
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv  (12,917 genes)
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.scores_summary.png
[genomics_scoring] Written → ../results/gene_score/SBC4_SBC10_SBC23.variant_scores.png


## Analysis

In [4]:
df_scored               = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.variant.tsv", sep="\t")
df_scored_gene_max      = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv", sep="\t")
df_scored_gene_sum      = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv", sep="\t") 
df_scored_gene_max_norm = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv", sep="\t")
df_scored_gene_sum_norm = pd.read_csv("../results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv", sep="\t")

In [5]:
df_scored_gene_sum_norm["ann_gene_id"].value_counts

<bound method IndexOpsMixin.value_counts of 0                   LOC110435944
1                      TRNAM-CAU
2                     LOC8083563
3                     LOC8081850
4                     LOC8071174
                  ...           
12912    LOC110436047-LOC8082070
12913                TRNAS-AGA_1
12914     TRNAS-CGA_3-LOC8080401
12915                TRNAS-GCU_2
12916                TRNAS-GCU_4
Name: ann_gene_id, Length: 12917, dtype: object>

In [6]:
len(df_scored_gene_sum_norm["rank"].unique())

6483

In [7]:
df_scored_gene_sum_norm["gene_length"].isna().sum()

np.int64(5674)

# Integration of variant analysis and gene co-expression data

## Preliminary analysis (only SEN102)

In [8]:
ROOT    = Path("/Users/daffa/workspace/infobio")
ZD      = ROOT / "Sbi-r.c1-0" / "Sbi-r.v25-12.G21627-S807.combat_pca.subagging.z.d"
TF_NAC  = ROOT / "thesis" / "resources" / "TFDB" / "Sbi_TF_list_NAC.txt"

SEN102  = "8058001"          # bait — the ONLY gene id hard-coded here
# Candidate universe restricted to the NAC family (Olvera-Carrillo 2015: NAC TFs
# are part of the dPCD gene set). D (8084661) / chr6 stay absent — keep it that way.

assert (ZD / SEN102).exists(), "SEN102 neighborhood file missing"
print("resource :", ZD.name)
print("bait     : SEN102 =", SEN102)

resource : Sbi-r.v25-12.G21627-S807.combat_pca.subagging.z.d
bait     : SEN102 = 8058001


In [9]:
# --- Load the NAC TF universe (EGI ids) -----------------------------------------
# file format: <TF_ID>\t<EGI_GeneID>  (no header; EGI is "NA" where unmapped)
nac = pd.read_csv(
    TF_NAC, sep="\t", header=None, names=["TF_ID", "EGI"], dtype=str
)
tf_egi = nac.loc[nac["EGI"].str.strip() != "NA", "EGI"].str.strip()
tf_set = set(tf_egi)
print(f"NAC TF universe: {len(nac)} NAC TFs, {len(tf_set)} with EGI ids")

# --- Load SEN102's genome-wide co-expression neighborhood -----------------------
# file format: <partner_EGI>\t<coex_score>, sorted descending by score
sen = pd.read_csv(
    ZD / SEN102, sep="\t", header=None, names=["EGI", "coex_score"], dtype={"EGI": str}
)
sen["rank_genomewide"] = range(1, len(sen) + 1)   # 1-based rank within SEN102's list
N_total = len(sen)
print(f"SEN102 neighborhood: {N_total} partner genes")

NAC TF universe: 180 NAC TFs, 109 with EGI ids
SEN102 neighborhood: 21626 partner genes


In [10]:
# --- Restrict SEN102's neighborhood to the TF universe, rank --------------------
tf_coex = (
    sen[sen["EGI"].isin(tf_set)]
    .copy()
    .reset_index(drop=True)
)
# percentile = how high this TF sits within SEN102's full genome-wide neighborhood
tf_coex["percentile"] = 1 - tf_coex["rank_genomewide"] / N_total
tf_coex["rank_among_TFs"] = range(1, len(tf_coex) + 1)
tf_coex = tf_coex[["rank_among_TFs", "EGI", "coex_score", "rank_genomewide", "percentile"]]

n_present = len(tf_coex)
print(f"{n_present} / {len(tf_set)} TFs appear in SEN102's neighborhood "
      f"({n_present/len(tf_set):.0%})")
print(f"coex_score range among TFs: {tf_coex.coex_score.min():.3f} … "
      f"{tf_coex.coex_score.max():.3f}")

65 / 109 TFs appear in SEN102's neighborhood (60%)
coex_score range among TFs: -1.381 … 3.776


In [11]:
# --- Top TFs co-expressed with SEN102 (the preliminary deliverable) -------------
TOP = 10
tf_coex.head(TOP)

,rank_among_TFs,EGI,coex_score,rank_genomewide,percentile
0,1,8084681,3.7763,17,0.999214
1,2,8084661,2.3005,154,0.992879
2,3,8078577,1.8464,364,0.983168
3,4,8057477,1.7653,434,0.979932
4,5,8083682,1.6842,503,0.976741
5,6,8074593,1.4734,780,0.963932
6,7,8074719,1.4734,793,0.963331
7,8,8057801,1.4572,805,0.962776
8,9,8078572,1.2950,1124,0.948026
9,10,8062451,1.2950,1131,0.947702


## Integrating with more dPCD related genes
| Generic Gene name | Arabidopsis gene ID, as found in the dPCD paper | Homolog in sorghum |
|-------------------|-------------------------------------------------|--------------------|
| PASPA3 - Aspartic protease 3 | At4g04460 | LOC8058198	LOC110430277	LOC110433911 |
| CEP1 - Cysteine endopeptidase | At5g50260 | LOC8058001 |
| MC9 - Metacaspase | At5g04200 | N/A |
| SCPL48 - Serine carboxypeptidase-like | At3g45010 | LOC8074232 |
| BFN1 - Bifunctional nuclease | At1g11190 | N/A |
| RNS3 - T2 Ribonuclease | At1g26820 | LOC8063515	LOC8063317	LOC8077825 | 

- Based on the Orvela-Carillo (2015) paper. Interestingly, the paper also mentioned that NAC family TFs are also included in this genes set. Utilizing this information, in the TF gene filtering, we will only take into account NAC family TF from the 2000 genes.
- Homolog(s) in sorghum is detected by ATTED-II built in orthologs tab

### Seed-set hygiene: is the dPCD seed one co-expression module?

Before aggregating co-expression across the 8 dPCD homologs, check they actually
co-express. We compute the **symmetric Mutual-Rank** between every seed pair
(`MR = sqrt(r_fwd · r_rev)`, lower = tighter, out of ~21626 genes) and then
**exhaustively enumerate every subset that contains the validated bait CEP1/SEN102**
(2⁷ = 128 subsets). Each subset is scored by its mean pairwise MR and compared to a
**size-matched random null** (random gene k-subsets), giving a z-score and empirical
p — so "which seeds belong together" is a number, not an eyeballed dendrogram cut.

Still fully D-blind: only PCD-execution genes are touched; no `8084661` / chr6 / NAC.

In [12]:
import itertools

# --- dPCD seed genes (EGI). CEP1/SEN102 is the original validated bait. ----------
SEEDS = {
    "8058198":   "PASPA3-a", "110430277": "PASPA3-b", "110433911": "PASPA3-c",
    "8058001":   "CEP1/SEN102", "8074232": "SCPL48",
    "8063515":   "RNS3-a", "8063317": "RNS3-b", "8077825": "RNS3-c",
}
ANCHOR = "8058001"                      # CEP1/SEN102 — present in every subset tested
RNG    = np.random.default_rng(0)

def load_ranks(egi):
    """{partner_egi: 1-based rank within egi's genome-wide neighborhood}, N_total"""
    d = {}
    with open(ZD / egi) as fh:
        for i, line in enumerate(fh, 1):
            g, _ = line.split()
            d[g] = i
    return d, i

rank, Nrec = {}, {}
for g in SEEDS:
    rank[g], Nrec[g] = load_ranks(g)
ALL_GENES = list(rank[ANCHOR].keys()) + [ANCHOR]      # genome-wide universe

def smr(a, b, rk):
    ra = rk[a].get(b, Nrec.get(a, len(rk[a])) + 1)
    rb = rk[b].get(a, Nrec.get(b, len(rk[b])) + 1)
    return np.sqrt(ra * rb)                            # symmetric Mutual Rank

def coherence(members, rk):                            # mean pairwise MR (lower = tighter)
    return float(np.mean([smr(a, b, rk) for a, b in itertools.combinations(members, 2)]))

# --- size-matched random null: load a gene pool once, draw random k-subsets -------
POOL_SIZE, N_PERM = 300, 3000
pool  = list(RNG.choice(ALL_GENES, size=POOL_SIZE, replace=False))
prank = {}
for g in pool:
    prank[g], Nrec[g] = load_ranks(g)

null = {k: np.array([coherence(RNG.choice(pool, size=k, replace=False), prank)
                     for _ in range(N_PERM)]) for k in range(2, len(SEEDS) + 1)}
null_mean = {k: v.mean() for k, v in null.items()}
null_std  = {k: v.std()  for k, v in null.items()}

# --- exhaustive enumeration of every CEP1-containing subset (size >= 2) -----------
others = [g for g in SEEDS if g != ANCHOR]
rows = []
for r in range(1, len(others) + 1):
    for combo in itertools.combinations(others, r):
        members = (ANCHOR,) + combo
        k, obs  = len(members), coherence((ANCHOR,) + combo, rank)
        rows.append({
            "k": k,
            "members": ", ".join(SEEDS[m] for m in members),
            "mean_pairwise_MR": round(obs, 1),
            "z_vs_random": round((obs - null_mean[k]) / null_std[k], 2),   # neg = tighter
            "emp_p": round((np.sum(null[k] <= obs) + 1) / (N_PERM + 1), 4),
        })

seed_subsets = pd.DataFrame(rows).sort_values("z_vs_random").reset_index(drop=True)
print("Most coherent CEP1-containing seed subsets (lowest z = tightest vs size-matched random):\n")
seed_subsets.head(12)

Most coherent CEP1-containing seed subsets (lowest z = tightest vs size-matched random):



,k,members,mean_pairwise_MR,z_vs_random,emp_p
0,4,"CEP1/SEN102, PASPA3-a, RNS3-b, RNS3-c",2478.6,-3.27,0.0013
1,5,"CEP1/SEN102, PASPA3-a, RNS3-a, RNS3-b, RNS3-c",4826.3,-3.05,0.0033
2,4,"CEP1/SEN102, PASPA3-a, RNS3-a, RNS3-c",3343.8,-2.93,0.0037
3,3,"CEP1/SEN102, PASPA3-a, RNS3-c",853.4,-2.76,0.0010
4,3,"CEP1/SEN102, RNS3-b, RNS3-c",1899.0,-2.47,0.0050
5,5,"CEP1/SEN102, PASPA3-a, PASPA3-c, RNS3-b, RNS3-c",6234.5,-2.33,0.0160
6,5,"CEP1/SEN102, PASPA3-b, PASPA3-c, SCPL48, RNS3-b",6503.8,-2.19,0.0210
7,6,"CEP1/SEN102, PASPA3-a, PASPA3-c, RNS3-a, RNS3-...",7081.2,-2.18,0.0250
8,4,"CEP1/SEN102, RNS3-a, RNS3-b, RNS3-c",5340.4,-2.14,0.0237
9,3,"CEP1/SEN102, RNS3-a, RNS3-c",3488.0,-2.03,0.0267


In [13]:
# Where the eyeballed "Module A" {CEP1, RNS3-c, PASPA3-a} lands, and the global optimum.
core = {"CEP1/SEN102", "RNS3-c", "PASPA3-a"}
is_core = seed_subsets["members"].apply(lambda s: set(s.split(", ")) == core)
print("Eyeballed 'Module A' {CEP1, RNS3-c, PASPA3-a}:")
print(seed_subsets[is_core].to_string(index=False))
print(f"\nGlobal best subset by size-normalised z: "
      f"{seed_subsets.iloc[0]['members']}  "
      f"(z={seed_subsets.iloc[0]['z_vs_random']}, p={seed_subsets.iloc[0]['emp_p']})")

# Which seeds never help the CEP1 module? Mean z of subsets containing each seed.
contrib = {SEEDS[g]: seed_subsets[seed_subsets["members"].str.contains(SEEDS[g], regex=False)]["z_vs_random"].mean()
           for g in others}
print("\nMean z of all CEP1-subsets containing each candidate seed (more negative = pulls the module tighter):")
for name, mz in sorted(contrib.items(), key=lambda kv: kv[1]):
    print(f"  {name:10s} {mz:6.2f}")

Eyeballed 'Module A' {CEP1, RNS3-c, PASPA3-a}:
 k                       members  mean_pairwise_MR  z_vs_random  emp_p
 3 CEP1/SEN102, PASPA3-a, RNS3-c             853.4        -2.76  0.001

Global best subset by size-normalised z: CEP1/SEN102, PASPA3-a, RNS3-b, RNS3-c  (z=-3.27, p=0.0013)

Mean z of all CEP1-subsets containing each candidate seed (more negative = pulls the module tighter):
  RNS3-b      -1.41
  RNS3-c      -1.33
  PASPA3-c    -1.01
  RNS3-a      -1.01
  PASPA3-a    -0.99
  PASPA3-b    -0.79
  SCPL48      -0.73


## Multi-bait NAC ranking — module core {CEP1/SEN102, RNS3-c, PASPA3-a}

Upgrade of the single-bait (SEN102-only) ranking to the 3-gene coherent core.
For each NAC TF `t` and each seed `s` we take the **symmetric Mutual Rank**
`MR(t,s) = sqrt(r_fwd · r_rev)`, then aggregate across the three seeds by
**rank product** `RP(t) = (∏_s MR(t,s))^(1/3)` (the geometric-mean generalization of
the frozen single-bait MR). Lower `RP` = co-expressed with the *whole* module.

> **Blindness note.** This cell crosses the D_GENE_COEX §6.2 wall on purpose: it both
> restricts to the NAC family and queries D (`8084661`). Per §2.3 the NAC restriction is
> a *disclosed secondary* view — the blind primary run must use the full ~1800 TF
> universe (one-line change: swap the NAC list for `Sbi_TF_gene_ids_EGI.txt`).

In [14]:
def rank_nac_tfs(baits, nac_path=ROOT / "thesis/resources/TFDB/Sbi_TF_list_NAC.txt"):
    """Rank NAC TFs by co-expression with one or more bait genes.

    baits : list[str] of bait EGI ids (e.g. ["8058001", "8077825", "8058198"]).
    Returns a DataFrame sorted ascending by RP (rank product of symmetric MR over
    all baits; lower = co-expressed with the whole bait set). One MR_<bait> column
    per bait, plus mean_z_fwd. A single bait reduces to the plain symmetric MR.
    """
    baits = list(baits)

    def load_scored(egi):                       # {partner: (rank, coex_z)}, N_total
        d = {}
        with open(ZD / egi) as fh:
            for i, line in enumerate(fh, 1):
                g, s = line.split()
                d[g] = (i, float(s))
        return d, i

    brank, bN = {}, {}
    for b in baits:
        brank[b], bN[b] = load_scored(b)

    nac     = pd.read_csv(nac_path, sep="\t", header=None, names=["TF_ID", "EGI"], dtype=str)
    egi_col = nac["EGI"].fillna("NA").str.strip()
    tf_egi  = sorted(set(egi_col[egi_col != "NA"]) - set(baits))   # a bait can't rank itself

    rows = []
    for t in tf_egi:
        try:
            trank, tN = load_scored(t)
        except FileNotFoundError:
            trank, tN = {}, max(bN.values())
        mrs, zfwd, row = [], [], {"EGI": t}
        for b in baits:
            r_fwd, z = brank[b].get(t, (bN[b] + 1, np.nan))
            r_rev    = trank.get(b, (tN + 1, np.nan))[0]
            mr = np.sqrt(r_fwd * r_rev)
            mrs.append(mr); zfwd.append(z)
            row[f"MR_{b}"] = round(mr, 0)
        row["RP"]         = round(float(np.prod(mrs) ** (1.0 / len(mrs))), 0)
        row["mean_z_fwd"] = round(np.nanmean(zfwd), 2)
        rows.append(row)

    out = pd.DataFrame(rows).sort_values("RP").reset_index(drop=True)
    out.insert(0, "rank", range(1, len(out) + 1))
    cols = ["rank", "EGI", "RP", "mean_z_fwd"] + [f"MR_{b}" for b in baits]
    return out[cols]


# demo: reproduce the 3-bait module ranking
ranked = rank_nac_tfs(["8058001", "8077825", "8058198"])   # CEP1/SEN102, RNS3-c, PASPA3-a
ranked.head(12)

/var/folders/9c/n04f21k12dl1pb6d9d3zlsz80000gn/T/ipykernel_27763/1261707027.py:41: RuntimeWarning: Mean of empty slice
  row["mean_z_fwd"] = round(np.nanmean(zfwd), 2)


,rank,EGI,RP,mean_z_fwd,MR_8058001,MR_8077825,MR_8058198
0,1,8084681,154.0,2.38,12.0,87.0,3600.0
1,2,8080831,717.0,1.54,1615.0,2278.0,100.0
2,3,8076977,1362.0,1.14,10032.0,35.0,7152.0
3,4,8074593,1510.0,1.17,853.0,3329.0,1213.0
4,5,8072366,1830.0,1.06,2377.0,251.0,10262.0
5,6,8067634,2063.0,1.04,1266.0,2017.0,3436.0
6,7,8079825,2070.0,1.14,1818.0,3447.0,1414.0
7,8,8084661,2162.0,0.61,120.0,4184.0,20078.0
8,9,8074719,2839.0,0.79,819.0,2506.0,11147.0
9,10,8057477,2925.0,0.80,531.0,4391.0,10729.0


## Are the nominated NAC TFs captured in the VCF? (genomics cross-check)

Join the co-expression ranking to the per-gene genomics scores
(`df_scored_gene_*` = the `SBC4_SBC10_SBC23` variant group, keyed by `ann_gene_id`
= `LOC<EGI>`). `in_VCF_group = False` means the gene carries **no variant in this
sharing group** — note this is group-specific, not "no variant anywhere".

In [15]:
def coex_vs_genomics(coex_df, gene_tables, top=12):
    """Join a co-expression ranking (column 'EGI') to per-gene genomics score tables.
    gene_tables: dict {tag: df_scored_gene_*}, each keyed by ann_gene_id = LOC<EGI>.
    """
    m = coex_df.head(top).copy()
    m["LOC"] = "LOC" + m["EGI"].astype(str)
    for tag, gt in gene_tables.items():
        cols = gt[["ann_gene_id", "score", "percentile", "rank"]].rename(
            columns={"score": f"{tag}_score", "percentile": f"{tag}_pct", "rank": f"{tag}_rank"})
        m = m.merge(cols, left_on="LOC", right_on="ann_gene_id", how="left").drop(columns="ann_gene_id")
    first_tag = next(iter(gene_tables))
    m.insert(2, "in_VCF_group", m[f"{first_tag}_pct"].notna())
    return m

cross = coex_vs_genomics(
    ranked,                                              # 3-bait NAC ranking from the cell above
    {"sum": df_scored_gene_sum_norm, "max": df_scored_gene_max_norm},
)
show_cols = ["rank", "EGI", "in_VCF_group",
             "sum_pct", "sum_rank", "max_pct", "max_rank"]
print("Top NAC TFs (3-bait co-expression) vs genomics score in SBC4_SBC10_SBC23:")
cross[show_cols]

Top NAC TFs (3-bait co-expression) vs genomics score in SBC4_SBC10_SBC23:


,rank,EGI,in_VCF_group,sum_pct,sum_rank,max_pct,max_rank
0,1,8084681,False,NaN,NaN,NaN,NaN
1,2,8080831,False,NaN,NaN,NaN,NaN
2,3,8076977,True,74.264807,1812.0,59.933729,2071.0
3,4,8074593,False,NaN,NaN,NaN,NaN
4,5,8072366,False,NaN,NaN,NaN,NaN
5,6,8067634,True,86.952920,938.0,93.013944,474.0
6,7,8079825,False,NaN,NaN,NaN,NaN
7,8,8084661,False,NaN,NaN,NaN,NaN
8,9,8074719,False,NaN,NaN,NaN,NaN
9,10,8057477,False,NaN,NaN,NaN,NaN


# Integration of DMR and gene co-expression data

In [16]:
df_dmr_score = pd.read_csv("/Users/daffa/workspace/infobio/thesis/analysis/DMR/dgene_promoter_candidates.tsv")

In [ ]:
def coex_vs_dmr(coex_df, gene_tables, top=12):
    """Join a co-expression ranking (column 'EGI') to per-gene DMR methylation mean difference tables.
    gene_table: df_dmr_score, each keyed by ann_gene_id = LOC<EGI>.
    """
    m = coex_df.head(top).copy()
    m["LOC"] = "LOC" + m["EGI"].astype(str)
    for tag, gt in gene_tables.items():
        cols = gt[["ann_gene_id", "score", "percentile", "rank"]].rename(
            columns={"score": f"{tag}_score", "percentile": f"{tag}_pct", "rank": f"{tag}_rank"})
        m = m.merge(cols, left_on="LOC", right_on="ann_gene_id", how="left").drop(columns="ann_gene_id")
    first_tag = next(iter(gene_tables))
    m.insert(2, "in_VCF_group", m[f"{first_tag}_pct"].notna())
    return m

print("Top NAC TFs (3-bait co-expression) vs DMR different mehylation rate in SBC4_SBC10_SBC23:")
